# Probability_Model — train (HGB per horizon)

One HistGradientBoosting ternary model per horizon $h \in \{3,10,30\}$.

**Data:** prefers `data/ES_1m.csv` (`time,open,high,low,close,volume`). If missing, falls back to synthetic bars so the loop still runs on this machine.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

# Repo root on sys.path when the kernel cwd is notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pm.bars import load_ohlcv, make_synthetic_ohlcv
from pm.pipeline import load_config, describe_readiness
from pm.train import run_train

CONFIG = ROOT / "configs" / "local_es.yaml"
ES_PATH = ROOT / "data" / "ES_1m.csv"
ARTIFACTS = ROOT / "artifacts" / "notebook_run"

cfg = load_config(CONFIG)
print(json.dumps(describe_readiness(cfg), indent=2))
print("config:", CONFIG)

{
  "horizons": [
    3,
    10,
    30
  ],
  "state_ids": [
    "ret_atr_1",
    "ret_atr_3",
    "ret_atr_5",
    "ret_atr_15",
    "ret_atr_30",
    "ret_atr_60",
    "atr_14",
    "atr_60",
    "atr_240",
    "atr_ratio_14_60",
    "atr_ratio_60_240",
    "rv_14",
    "rv_60",
    "body_atr",
    "upper_wick_atr",
    "lower_wick_atr",
    "range_atr",
    "close_pos",
    "dir_consist_15",
    "dist_ema_10",
    "dist_ema_30",
    "dist_ema_60",
    "ema_slope_10",
    "ema_slope_60",
    "eff_30",
    "rvol_5",
    "rvol_30",
    "vol_z_60",
    "vol_chg",
    "vol_x_ret"
  ],
  "target_id": "ternary_displacement",
  "registered_states": [
    "atr_14",
    "atr_240",
    "atr_60",
    "atr_ratio_14_60",
    "atr_ratio_60_240",
    "body_atr",
    "close_pos",
    "dir_consist_15",
    "dist_ema_10",
    "dist_ema_30",
    "dist_ema_60",
    "eff_30",
    "ema_slope_10",
    "ema_slope_60",
    "lower_wick_atr",
    "range_atr",
    "ret_atr_1",
    "ret_atr_15",
    "ret_atr_3"

## Load bars

Drop a local Databento/Quanqora export at `data/ES_1m.csv` (gitignored). Without it we smoke-train on synthetic 1m bars.

In [2]:
if ES_PATH.exists():
    bars = load_ohlcv(ES_PATH)
    data_source = f"file:{ES_PATH.relative_to(ROOT)}"
else:
    # Enough bars for ATR warmup + purge=30 + holdout
    bars = make_synthetic_ohlcv(n=5000, seed=42)
    data_source = "synthetic(n=5000, seed=42)"

print(data_source)
print(f"rows={len(bars):,}  range={bars.index.min()} → {bars.index.max()}")
bars.head()

synthetic(n=5000, seed=42)
rows=5,000  range=2024-01-01 00:00:00+00:00 → 2024-01-04 11:19:00+00:00


,open,high,low,close,volume
time,,,,,
2024-01-01 00:00:00+00:00,100.015236,100.030578,100.004012,100.015236,611.937082
2024-01-01 00:01:00+00:00,100.015236,100.036818,99.958714,99.963237,230.498851
2024-01-01 00:02:00+00:00,99.963237,100.028474,99.956902,100.000759,819.059500
2024-01-01 00:03:00+00:00,100.000759,100.075415,99.992583,100.047787,542.089310
2024-01-01 00:04:00+00:00,100.047787,100.049881,99.935134,99.950236,609.816602


## Train

Time holdout (`holdout_frac=0.2`) + purge (`purge=30`). Artifacts → `artifacts/notebook_run/`.

In [3]:
result = run_train(cfg, bars=bars, artifacts_dir=ARTIFACTS)

summary = {
    "data_source": data_source,
    "backend": result["backend"],
    "symbol": result["symbol"],
    "purge": result["purge"],
    "holdout_frac": result["holdout_frac"],
    "horizons": {
        h: {
            "n_train": v["n_train"],
            "n_holdout": v["n_holdout"],
            "log_loss": v["metrics"]["log_loss"],
            "brier_mean": v["metrics"]["brier_mean"],
            "accuracy": v["metrics"]["accuracy"],
            "class_frac_true": v["metrics"]["class_frac_true"],
        }
        for h, v in result["horizons"].items()
    },
}
print(json.dumps(summary, indent=2))
list(ARTIFACTS.iterdir())

{
  "data_source": "synthetic(n=5000, seed=42)",
  "backend": "hgb",
  "symbol": "ES",
  "purge": 30,
  "holdout_frac": 0.2,
  "horizons": {
    "3": {
      "n_train": 3776,
      "n_holdout": 952,
      "log_loss": 1.1062346238180423,
      "brier_mean": 0.226588167905415,
      "accuracy": 0.4023109243697479,
      "class_frac_true": {
        "down": 0.453781512605042,
        "flat": 0.11029411764705882,
        "up": 0.43592436974789917
      }
    },
    "10": {
      "n_train": 3770,
      "n_holdout": 951,
      "log_loss": 1.0527769888291827,
      "brier_mean": 0.21909558838143117,
      "accuracy": 0.45320715036803366,
      "class_frac_true": {
        "down": 0.5131440588853838,
        "flat": 0.0683491062039958,
        "up": 0.4185068349106204
      }
    },
    "30": {
      "n_train": 3754,
      "n_holdout": 947,
      "log_loss": 1.1617635979875454,
      "brier_mean": 0.242754497957944,
      "accuracy": 0.42133051742344246,
      "class_frac_true": {
        "dow

[PosixPath('/workspace/artifacts/notebook_run/model_h30.joblib'),
 PosixPath('/workspace/artifacts/notebook_run/model_h3.joblib'),
 PosixPath('/workspace/artifacts/notebook_run/metrics.json'),
 PosixPath('/workspace/artifacts/notebook_run/model_h10.joblib')]

## Metrics table

In [4]:
rows = []
for h, v in result["horizons"].items():
    m = v["metrics"]
    rows.append(
        {
            "horizon": int(h),
            "n_train": v["n_train"],
            "n_holdout": v["n_holdout"],
            "log_loss": m["log_loss"],
            "brier_mean": m["brier_mean"],
            "accuracy": m["accuracy"],
        }
    )
pd.DataFrame(rows).set_index("horizon").round(4)

,n_train,n_holdout,log_loss,brier_mean,accuracy
horizon,,,,,
3,3776,952,1.1062,0.2266,0.4023
10,3770,951,1.0528,0.2191,0.4532
30,3754,947,1.1618,0.2428,0.4213
